<a href="https://colab.research.google.com/github/hig3r/mva/blob/2026/E04/mva_2026_E04_02_distribution_of_regression_coefficients.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mva_2026_E04_02_distribution_of_regression_coefficients 回帰係数の推定量の分布

このNotebookは，（2年生段階で）みなさんが中身を理解して分析のために使ったり書いたりするコードではなく，実行して結果を観察するためのものです．ありません．

In [ ]:
import numpy as np # 数値計算ライブラリ
import pandas as pd # データ解析ライブラリ
import matplotlib.pyplot as plt # グラフ描画ライブラリ
%matplotlib inline

from sklearn.linear_model import LinearRegression # scikit-learn の線形重回帰ライブラリ

!pip install japanize_matplotlib
import japanize_matplotlib # 日本語フォント

人工的に，線形重回帰モデルに従うサイズ$n$の標本を$m$個作ります．

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# パラメータ設定
n = 100         # 1サンプルあたりのデータ数
m = 1000        # サミュレーション回数（サンプルの数）
true_beta0 = 2.0  # 仏のみ知る真の切片
true_beta1 = 3.5  # 仏のみ知る真の回帰係数
noise_sd = 2.0    # 誤差項の標準偏差

# 推定値を格納するリスト
estimated_beta0 = []
estimated_beta1 = []

# 乱数のシードを固定（再現性のため）
np.random.seed(42)

# 固定された説明変数 X を生成（ここでは0から10の一様分布から100点）
X_fixed = np.random.uniform(0, 10, size=(n, 1))

# シミュレーションループ for(i=0;i<m;i++)相当
for i in range(m):
    # 誤差項の生成
    epsilon = np.random.normal(loc=0.0, scale=noise_sd, size=(n, 1))
    # 被説明変数 Y の生成
    y_sim = true_beta0 + true_beta1 * X_fixed + epsilon

    # 回帰モデルのフィッティング
    model = LinearRegression()
    model.fit(X_fixed, y_sim)

    # 推定された係数の保存
    estimated_beta0.append(model.intercept_[0])
    estimated_beta1.append(model.coef_[0][0])

# 結果をDataFrameにまとめる
results_df = pd.DataFrame({
    'beta0_estimate': estimated_beta0,
    'beta1_estimate': estimated_beta1
})

In [ ]:
display(results_df.head())

確率変数の実測値$\epsilon_1,\dots,\epsilon_n$ したがって，目的変数の実測値$y_1,\dots,y_n$を$m$回繰り返して取得し，それに対して，回帰係数$(\hat{\beta}_0,\hat{\beta}_1)$ を $m$ 回推定(estimate)します．この$m$個は2次元正規分布$N(\beta,\sigma^2(X^\top X)^{-1})$ にしたがって分布するはずなのでした．なってる？ここから$\sigma^2$を推定することもできるはず．

In [ ]:
# (beta0, beta1) の推定量に関する散布図を描画
plt.figure(figsize=(8, 6))
plt.scatter(results_df['beta0_estimate'], results_df['beta1_estimate'], color='purple', alpha=0.4, edgecolors='none', label=r'$(\hat{\beta}_0, \hat{\beta}_1)$の推定値')

# 真の値をプロット
plt.plot(true_beta0, true_beta1, 'r*', markersize=15, label=f'仏のみ知る真の値({true_beta0}, {true_beta1})')

# ラベルやタイトルの設定
plt.title(r'$(\hat{\beta}_0, \hat{\beta}_1)$の推定値の同時分布')
plt.xlabel(r'$\beta_0$ (切片)の推定値)')
plt.ylabel(r'$\beta_1$ (回帰係数)の推定値')
plt.axvline(true_beta0, color='red', linestyle='--', alpha=0.5)
plt.axhline(true_beta1, color='red', linestyle='--', alpha=0.5)

plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# 推定値の要約統計量を計算・表示
summary_stats = results_df.describe()

# より見やすくするために、真の値との比較行を追加します
true_values = pd.Series({'beta0_estimate': true_beta0, 'beta1_estimate': true_beta1}, name='true_value')
summary_stats = pd.concat([summary_stats, true_values.to_frame().T])

display(summary_stats)
# mean が標本平均値
# std が標本標準偏差 standard deviation

In [ ]:
# 推定値の分散共分散行列（Covariance Matrix）を計算・表示
covariance_matrix = results_df.cov()
display(covariance_matrix)

各変数の周辺分布のヒストグラムを見てみましょう．多変量正規分布の周辺分布はまた正規分布になってるはずですが．

In [ ]:
# 推定値の分布をプロット
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 切片 beta0 の分布
axes[0].hist(results_df['beta0_estimate'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(true_beta0, color='red', linestyle='--', linewidth=2, label=fr'真の $\hat{{\beta_0}}$ ({true_beta0})')
axes[0].set_title(r'$\hat{\beta_0}$ (回帰係数)')
axes[0].set_xlabel('推定値')
axes[0].set_ylabel('頻度')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# 傾き beta1 の分布
axes[1].hist(results_df['beta1_estimate'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1].axvline(true_beta1, color='red', linestyle='--', linewidth=2, label=fr'真の $\hat{{\beta_1}}$ ({true_beta1})')
axes[1].set_title(r'$\hat{\beta_1}$ (回帰係数)')
axes[1].set_xlabel('推定値')
axes[1].set_ylabel('頻度')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()